# 12 — Universality and Data Collapse

**Universal transition curves for distributed residue consistency**

Notebook 11 showed that projection thresholds behave like finite-size logistic phase transitions.

Notebook 12 asks:

```text
Do different graph sizes share one universal transition curve after rescaling?
```

Canonical collapse variable:

```text
z = (link_noise - noise_crit(N)) / sigma(N)
```

Then:

```text
p_required ≈ F(z)
```

where `F` is a bounded transition profile.

## Outputs

```text
figures/universal_collapse_comparison.png
figures/universal_curve_residuals.png
figures/collapse_quality_by_N.png
figures/leave_one_size_out.png

results/collapse_data.csv
results/universal_curve_comparison.csv
results/best_universal_model.json
results/collapse_quality_by_N.csv
results/leave_one_size_out.csv
results/universality_summary.json

docs/notebook_12_universality_and_data_collapse.md
```

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import curve_fit
    from scipy.special import erf
except ImportError:
    !pip -q install scipy
    from scipy.optimize import curve_fit
    from scipy.special import erf

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")

for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PHASE_LOCK_THRESHOLD = 24 / 25

print("Ready.")
print(f"phase-lock threshold = {PHASE_LOCK_THRESHOLD:.3f}")

## 1. Load Notebook 11 outputs

Inputs:

```text
results/projection_threshold_scaling.csv
results/logistic_fit_by_N.csv
```

If missing, this notebook creates demonstration data consistent with Notebook 11.

In [ ]:
threshold_path = RESULTS_DIR / "projection_threshold_scaling.csv"
fit_path = RESULTS_DIR / "logistic_fit_by_N.csv"

if threshold_path.exists():
    df = pd.read_csv(threshold_path)
    print(f"loaded: {threshold_path}")
else:
    print("projection_threshold_scaling.csv not found; using demonstration threshold data.")
    graph_sizes = [12, 20, 32]
    noise_values = np.array([0.00, 0.03, 0.05, 0.07, 0.09, 0.12, 0.15])
    observed_thresholds = {
        12: [0.0, 0.0, 0.05, 0.25, 0.60, 0.72, 0.84],
        20: [0.0, 0.0, 0.10, 0.32, 0.68, 0.80, 1.00],
        32: [0.0, 0.0, 0.00, 0.18, 0.50, 0.70, 0.86],
    }
    rows = []
    for N in graph_sizes:
        for noise, p_req in zip(noise_values, observed_thresholds[N]):
            rows.append({"n_modules": N, "link_noise": noise, "p_required": p_req})
    df = pd.DataFrame(rows)

required = {"n_modules", "link_noise", "p_required"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns from threshold data: {missing}")

df = df.copy()
df["n_modules"] = df["n_modules"].astype(int)
df["link_noise"] = df["link_noise"].astype(float)
df["p_required"] = df["p_required"].astype(float)

if fit_path.exists():
    fit_df = pd.read_csv(fit_path)
    print(f"loaded: {fit_path}")
else:
    print("logistic_fit_by_N.csv not found; fitting logistic parameters here.")

    def logistic(noise, noise_crit, sigma):
        sigma = max(float(sigma), 1e-4)
        return 1 / (1 + np.exp(-(noise - noise_crit) / sigma))

    fit_rows = []
    for N, group in df.groupby("n_modules"):
        x = group["link_noise"].to_numpy(dtype=float)
        y = group["p_required"].to_numpy(dtype=float)
        try:
            popt, _ = curve_fit(
                logistic,
                x,
                y,
                p0=[0.08, 0.02],
                bounds=([0.0, 1e-4], [1.0, 1.0]),
                maxfev=10000,
            )
            noise_crit, sigma = popt
        except Exception:
            noise_crit = float(x[np.argmin(np.abs(y - 0.5))])
            sigma = 0.03

        fit_rows.append(
            {
                "n_modules": int(N),
                "noise_crit": float(noise_crit),
                "sigma": float(sigma),
            }
        )

    fit_df = pd.DataFrame(fit_rows)
    fit_df.to_csv(fit_path, index=False)
    print(f"saved: {fit_path}")

required_fit = {"n_modules", "noise_crit", "sigma"}
missing_fit = required_fit - set(fit_df.columns)
if missing_fit:
    raise ValueError(f"Missing required columns from fit data: {missing_fit}")

fit_df["n_modules"] = fit_df["n_modules"].astype(int)
fit_df["noise_crit"] = fit_df["noise_crit"].astype(float)
fit_df["sigma"] = fit_df["sigma"].astype(float)

print(df.head())
print(fit_df)

## 2. Build collapse data

For each observation:

```text
z = (link_noise - noise_crit(N)) / sigma(N)
```

In [ ]:
collapse_df = df.merge(
    fit_df[["n_modules", "noise_crit", "sigma"]],
    on="n_modules",
    how="left",
)

collapse_df["sigma_safe"] = collapse_df["sigma"].clip(lower=1e-4)
collapse_df["z"] = (
    collapse_df["link_noise"] - collapse_df["noise_crit"]
) / collapse_df["sigma_safe"]

collapse_df = collapse_df.dropna(subset=["z", "p_required"]).copy()

collapse_path = RESULTS_DIR / "collapse_data.csv"
collapse_df.to_csv(collapse_path, index=False)

print(collapse_df.head())
print(f"saved: {collapse_path}")

## 3. Candidate universal transition curves

Compare bounded transition profiles:

```text
logistic(z)
tanh(z)
erf(z)
```

We fit a small global correction:

```text
p = F(a z + b)
```

In [ ]:
def logistic_curve(x):
    return 1 / (1 + np.exp(-x))

def tanh_curve(x):
    return 0.5 * (1 + np.tanh(x / 2))

def erf_curve(x):
    return 0.5 * (1 + erf(x / np.sqrt(2)))

MODEL_FUNCS = {
    "logistic": logistic_curve,
    "tanh": tanh_curve,
    "erf": erf_curve,
}

def corrected_curve(z, a, b, model_name):
    return MODEL_FUNCS[model_name](a * z + b)

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    residual = y_pred - y_true
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    mae = float(np.mean(np.abs(residual)))

    denom = np.sum((y_true - np.mean(y_true)) ** 2)
    if denom == 0:
        r2 = np.nan
    else:
        r2 = float(1 - np.sum(residual ** 2) / denom)

    return {"rmse": rmse, "mae": mae, "r2": r2}

## 4. Fit global universal curves

In [ ]:
z = collapse_df["z"].to_numpy(dtype=float)
y = collapse_df["p_required"].to_numpy(dtype=float)

comparison_rows = []
prediction_frames = []

for model_name in MODEL_FUNCS:
    def fit_func(z_input, a, b):
        return corrected_curve(z_input, a, b, model_name)

    try:
        popt, _ = curve_fit(
            fit_func,
            z,
            y,
            p0=[1.0, 0.0],
            bounds=([0.01, -10.0], [10.0, 10.0]),
            maxfev=10000,
        )
        a, b = popt
    except Exception:
        a, b = 1.0, 0.0

    y_pred = fit_func(z, a, b)
    m = metrics(y, y_pred)

    comparison_rows.append(
        {
            "model": model_name,
            "a": float(a),
            "b": float(b),
            **m,
        }
    )

    tmp = collapse_df.copy()
    tmp["model"] = model_name
    tmp["prediction"] = y_pred
    tmp["residual"] = tmp["prediction"] - tmp["p_required"]
    prediction_frames.append(tmp)

comparison_df = pd.DataFrame(comparison_rows).sort_values("rmse")
predictions_df = pd.concat(prediction_frames, ignore_index=True)

comparison_path = RESULTS_DIR / "universal_curve_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print(comparison_df)
print(f"saved: {comparison_path}")

## 5. Collapse comparison figure

Observed collapsed points plus best-fit universal profiles.

In [ ]:
z_grid = np.linspace(
    min(collapse_df["z"].min(), -6),
    max(collapse_df["z"].max(), 6),
    500,
)

plt.figure(figsize=(10, 6))

for N, group in collapse_df.groupby("n_modules"):
    plt.scatter(
        group["z"],
        group["p_required"],
        s=120,
        label=f"N={N}",
        alpha=0.85,
    )

for _, row in comparison_df.iterrows():
    model_name = row["model"]
    a = row["a"]
    b = row["b"]
    y_grid = corrected_curve(z_grid, a, b, model_name)

    linestyle = "-" if model_name == comparison_df.iloc[0]["model"] else "--"

    plt.plot(
        z_grid,
        y_grid,
        linewidth=3,
        linestyle=linestyle,
        label=f"{model_name} RMSE={row['rmse']:.3f}",
    )

plt.xlabel("collapsed variable z")
plt.ylabel("required projection success")
plt.title("Universal collapse comparison")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

collapse_fig = FIG_DIR / "universal_collapse_comparison.png"
plt.savefig(collapse_fig, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {collapse_fig}")

## 6. Residuals by model

In [ ]:
plt.figure(figsize=(10, 6))

models = comparison_df["model"].tolist()

for idx, model_name in enumerate(models):
    residuals = predictions_df[predictions_df["model"] == model_name]["residual"].to_numpy()
    x_jitter = np.random.default_rng(idx).normal(idx, 0.035, size=len(residuals))

    plt.scatter(
        x_jitter,
        residuals,
        s=80,
        alpha=0.75,
        label=model_name,
    )

plt.axhline(0, linestyle="--", linewidth=2)
plt.xticks(range(len(models)), models)
plt.ylabel("prediction - observed")
plt.title("Universal curve residuals")
plt.grid(True, alpha=0.3)
plt.tight_layout()

resid_fig = FIG_DIR / "universal_curve_residuals.png"
plt.savefig(resid_fig, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {resid_fig}")

## 7. Best model selection

In [ ]:
best = comparison_df.iloc[0].to_dict()

best_payload = {
    "best_model": best["model"],
    "a": float(best["a"]),
    "b": float(best["b"]),
    "rmse": float(best["rmse"]),
    "mae": float(best["mae"]),
    "r2": None if pd.isna(best["r2"]) else float(best["r2"]),
    "interpretation": "Bounded sigmoid-type transitions describe the collapse.",
}

best_path = RESULTS_DIR / "best_universal_model.json"
best_path.write_text(json.dumps(best_payload, indent=2), encoding="utf-8")

print(json.dumps(best_payload, indent=2))
print(f"saved: {best_path}")

## 8. Collapse quality by graph size

Measure RMSE per graph size using the best universal model.

In [ ]:
best_model = best_payload["best_model"]
best_a = best_payload["a"]
best_b = best_payload["b"]

best_pred = predictions_df[predictions_df["model"] == best_model].copy()

quality_rows = []

for N, group in best_pred.groupby("n_modules"):
    m = metrics(group["p_required"], group["prediction"])
    quality_rows.append({"n_modules": int(N), **m})

quality_df = pd.DataFrame(quality_rows)

quality_path = RESULTS_DIR / "collapse_quality_by_N.csv"
quality_df.to_csv(quality_path, index=False)

print(quality_df)
print(f"saved: {quality_path}")

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    quality_df["n_modules"].astype(str),
    quality_df["rmse"],
)

plt.xlabel("graph size N")
plt.ylabel("RMSE")
plt.title(f"Collapse quality by graph size ({best_model})")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()

quality_fig = FIG_DIR / "collapse_quality_by_N.png"
plt.savefig(quality_fig, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {quality_fig}")

## 9. Leave-one-size-out validation

Fit universal curve on two graph sizes, test on the third.

In [ ]:
l1o_rows = []

for held_out_N in sorted(collapse_df["n_modules"].unique()):
    train = collapse_df[collapse_df["n_modules"] != held_out_N].copy()
    test = collapse_df[collapse_df["n_modules"] == held_out_N].copy()

    z_train = train["z"].to_numpy(dtype=float)
    y_train = train["p_required"].to_numpy(dtype=float)

    z_test = test["z"].to_numpy(dtype=float)
    y_test = test["p_required"].to_numpy(dtype=float)

    for model_name in MODEL_FUNCS:
        def fit_func(z_input, a, b):
            return corrected_curve(z_input, a, b, model_name)

        try:
            popt, _ = curve_fit(
                fit_func,
                z_train,
                y_train,
                p0=[1.0, 0.0],
                bounds=([0.01, -10.0], [10.0, 10.0]),
                maxfev=10000,
            )
            a, b = popt
        except Exception:
            a, b = 1.0, 0.0

        y_pred = fit_func(z_test, a, b)
        m = metrics(y_test, y_pred)

        l1o_rows.append(
            {
                "held_out_N": int(held_out_N),
                "model": model_name,
                "a": float(a),
                "b": float(b),
                **m,
            }
        )

l1o_df = pd.DataFrame(l1o_rows).sort_values(["held_out_N", "rmse"])

l1o_path = RESULTS_DIR / "leave_one_size_out.csv"
l1o_df.to_csv(l1o_path, index=False)

print(l1o_df)
print(f"saved: {l1o_path}")

In [ ]:
plt.figure(figsize=(9, 5))

for model_name, group in l1o_df.groupby("model"):
    group = group.sort_values("held_out_N")
    plt.plot(
        group["held_out_N"],
        group["rmse"],
        marker="o",
        linewidth=2,
        label=model_name,
    )

plt.xlabel("held-out graph size N")
plt.ylabel("test RMSE")
plt.title("Leave-one-size-out validation")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

l1o_fig = FIG_DIR / "leave_one_size_out.png"
plt.savefig(l1o_fig, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {l1o_fig}")

## 10. Summary exports

In [ ]:
summary_payload = {
    "notebook": "12_universality_and_data_collapse.ipynb",
    "phase_lock_threshold": PHASE_LOCK_THRESHOLD,
    "collapse_variable": "z = (link_noise - noise_crit(N)) / sigma(N)",
    "best_model": best_payload,
    "core_claim": (
        "After rescaling by midpoint and width, threshold curves across graph sizes "
        "approximately collapse onto a bounded sigmoid-type transition profile."
    ),
    "figures": [
        "figures/universal_collapse_comparison.png",
        "figures/universal_curve_residuals.png",
        "figures/collapse_quality_by_N.png",
        "figures/leave_one_size_out.png",
    ],
    "results": [
        "results/collapse_data.csv",
        "results/universal_curve_comparison.csv",
        "results/best_universal_model.json",
        "results/collapse_quality_by_N.csv",
        "results/leave_one_size_out.csv",
        "results/universality_summary.json",
    ],
}

summary_path = RESULTS_DIR / "universality_summary.json"
summary_path.write_text(
    json.dumps(summary_payload, indent=2),
    encoding="utf-8",
)

doc_lines = [
    "# Notebook 12 — Universality and Data Collapse",
    "",
    "**Core claim:** finite-size threshold curves approximately collapse onto a shared bounded transition profile.",
    "",
    "Collapse variable:",
    "",
    "`z = (link_noise - noise_crit(N)) / sigma(N)`",
    "",
    f"Best model: `{best_payload['best_model']}`",
    "",
    "Outputs:",
    "",
    "- `figures/universal_collapse_comparison.png`",
    "- `figures/universal_curve_residuals.png`",
    "- `figures/collapse_quality_by_N.png`",
    "- `figures/leave_one_size_out.png`",
    "",
]

doc_path = DOCS_DIR / "notebook_12_universality_and_data_collapse.md"
doc_path.write_text("\n".join(doc_lines), encoding="utf-8")

print(json.dumps(summary_payload, indent=2))
print(f"saved: {summary_path}")
print(f"saved: {doc_path}")

## 11. Optional zip/export block for Colab

This cell bundles generated figures, results, and docs into one zip file.

Uncomment the last two lines when running in Google Colab to download the zip directly.

In [ ]:
import zipfile

zip_path = Path("notebook_12_outputs.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder_name in ["figures", "results", "docs"]:
        folder = Path(folder_name)
        if folder.exists():
            for path in folder.rglob("*"):
                if path.is_file():
                    z.write(path, path.as_posix())

print(f"created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))

## Final interpretation

Notebook 12 tests whether graph-size-specific threshold curves share a common transition profile.

Careful conclusion:

```text
Within this finite-size range, the collapse is consistent with a bounded sigmoid-type threshold curve.
```

This strengthens the phase-transition interpretation without overclaiming universality.